---

## 5. Modelling

### 5.1 Implementasi Algoritma Naive Bayes Multinomial

In [ ]:
import numpy as np

class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        n_samples, n_features = X.shape
        self._classes = np.unique(y)
        n_classes = len(self._classes)

        self._vocab = n_features
        self._word_counts = np.zeros((n_classes, n_features), dtype=np.float64)
        self._total_words = np.zeros(n_classes, dtype=np.float64)
        self._priors = np.zeros(n_classes, dtype=np.float64)
        self._likelihoods = np.zeros((n_classes, n_features), dtype=np.float64)

        for index, c in enumerate(self._classes):
            X_c = X[y == c]
            self._word_counts[index, :] = X_c.sum(axis=0)
            self._total_words[index] = X_c.sum()
            self._priors[index] = X_c.shape[0] / float(n_samples)
            self._likelihoods[index, :] = self._likelihood(index)

        return self

    def _likelihood(self, class_index):
        word_counts = self._word_counts[class_index]
        total_words = self._total_words[class_index]
        numerator = word_counts + self.alpha
        denominator = total_words + (self.alpha * self._vocab)
        return numerator / denominator

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        for index, c in enumerate(self._classes):
            prior = np.log(self._priors[index])
            posterior = np.sum(x * np.log(self._likelihoods[index]))
            posterior = posterior + prior
            posteriors.append(posterior)

        return self._classes[np.argmax(posteriors)]

    def predict_probabilities(self, X):
        X = np.asarray(X, dtype=np.float64)
        probabilities = []

        for x in X:
            log_posteriors = []
            for index, c in enumerate(self._classes):
                prior = np.log(self._priors[index])
                posterior = np.sum(x * np.log(self._likelihoods[index])) + prior
                log_posteriors.append(posterior)

            log_posteriors = np.array(log_posteriors)
            exp_scores = np.exp(log_posteriors - np.max(log_posteriors))
            probs = exp_scores / np.sum(exp_scores)
            probabilities.append(probs)

        return np.array(probabilities)

### 5.2 Pelatihan Model

In [ ]:
import pandas as pd

naive_bayes = NaiveBayes(alpha=1.0)
naive_bayes.fit(X_train, y_train)

ringkasan = pd.DataFrame({
    'Kelas': naive_bayes._classes,
    'Jumlah Dokumen Latih': [np.sum(y_train == c) for c in naive_bayes._classes],
    'Prior Probability': np.round(naive_bayes._priors, 4),
    'Prior': np.round(naive_bayes._priors * 100, 2),
    'Frekuensi Kata': np.round(naive_bayes._total_words, 2)
})

print(f"Jumlah Kelas Unik   : {len(naive_bayes._classes)} kelas")
print(f"Ukuran Vocabulary   : {naive_bayes._vocab} fitur kata unik\n")
print(ringkasan)

### 5.3 Prediksi Model pada Data Latih dan Data Uji

In [ ]:
y_pred_train = naive_bayes.predict(X_train)
y_pred_test = naive_bayes.predict(X_test)
y_prob_test = naive_bayes.predict_probabilities(X_test)

hasil_prediksi = pd.DataFrame({
    'Label Aktual': y_test[:10],
    'Hasil Prediksi': y_pred_test[:10],
    'Keyakinan Model': np.round(np.max(y_prob_test[:10], axis=1) * 100, 2)
})

print("10 Hasil Prediksi pada Data Uji:")
print(hasil_prediksi)

---

## 6. Evaluation

### 6.1 Implementasi Fungsi Evaluasi Multiclass

In [ ]:
def evaluate_multiclass(y_true, y_pred, classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n_classes = len(classes)

    cm = np.zeros((n_classes, n_classes), dtype=int)
    for i, c_actual in enumerate(classes):
        for j, c_pred in enumerate(classes):
            cm[i, j] = np.sum((y_true == c_actual) & (y_pred == c_pred))

    metrics_list = []
    for i, c in enumerate(classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        tn = np.sum(cm) - (tp + fp + fn)
        support = np.sum(cm[i, :])

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        metrics_list.append({'Kelas': c, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'Precision': round(precision, 4), 'Recall': round(recall, 4), 'F1-Score': round(f1, 4), 'Support': support})

    df_metrics = pd.DataFrame(metrics_list)

    accuracy = np.mean(y_true == y_pred)
    total_support = np.sum(df_metrics['Support'])

    summary = {
        'Accuracy': round(accuracy, 4), 'Macro Precision': round(df_metrics['Precision'].mean(), 4), 'Macro Recall': round(df_metrics['Recall'].mean(), 4), 'Macro F1-Score': round(df_metrics['F1-Score'].mean(), 4),
        'Weighted Precision': round(np.sum(df_metrics['Precision'] * df_metrics['Support']) / total_support, 4), 'Weighted Recall': round(np.sum(df_metrics['Recall'] * df_metrics['Support']) / total_support, 4), 'Weighted F1-Score': round(np.sum(df_metrics['F1-Score'] * df_metrics['Support']) / total_support, 4)
    }

    return cm, df_metrics, summary

### 6.2 Hasil Evaluasi Model pada Data Latih dan Data Uji

In [ ]:
cm_train, df_metrics_train, summary_train = evaluate_multiclass(y_train, y_pred_train, naive_bayes._classes)
cm_test, df_metrics_test, summary_test = evaluate_multiclass(y_test, y_pred_test, naive_bayes._classes)

ringkasan_evaluasi = pd.DataFrame({
    'Metrik Evaluasi': list(summary_test.keys()),
    'Data Latih': list(summary_train.values()),
    'Data Uji': list(summary_test.values()),
    'Data Uji (%)': [round(val * 100, 2) for val in summary_test.values()]
})

print("Perbandingan Performa Model:")
print(ringkasan_evaluasi.to_string(index=False))

print("\nDetail Evaluasi per Kelas pada Data Uji:")
print(df_metrics_test.to_string(index=False))

### 6.3 Visualisasi Confusion Matrix dan Metrik per Kelas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 5.5))

plt.subplot(1, 2, 1)
df_cm = pd.DataFrame(cm_test, index=naive_bayes._classes, columns=naive_bayes._classes)
sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix pada Data Uji', fontsize=12, fontweight='bold')
plt.xlabel('Hasil Prediksi')
plt.ylabel('Label Aktual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.subplot(1, 2, 2)
x_pos = np.arange(len(naive_bayes._classes))
width = 0.25

plt.bar(x_pos - width, df_metrics_test['Precision'], width=width, label='Precision', color='#2b5c8f')
plt.bar(x_pos, df_metrics_test['Recall'], width=width, label='Recall', color='#388e3c')
plt.bar(x_pos + width, df_metrics_test['F1-Score'], width=width, label='F1-Score', color='#f57c00')

plt.title('Precision, Recall, dan F1-Score per Kelas', fontsize=12, fontweight='bold')
plt.xlabel('Kelas Target')
plt.ylabel('Skor (0.0 - 1.0)')
plt.xticks(x_pos, naive_bayes._classes, rotation=45, ha='right')
plt.ylim(0, 1.1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

---

## 7. Graphical User Interface (GUI)

In [ ]:
import tkinter as tk

def ubah_teks_ke_vektor(teks):
    teks_bersih = "".join([c if c.isalpha() or c.isspace() else " " for c in teks.lower()])
    tokens = teks_bersih.split()

    if 'vectorizer' in globals() and hasattr(vectorizer, 'transform'):
        hasil = vectorizer.transform([teks_bersih])
        return hasil.toarray() if hasattr(hasil, 'toarray') else np.asarray(hasil, dtype=np.float64)

    daftar_kata = vocab if 'vocab' in globals() else (vocabulary if 'vocabulary' in globals() else X_train.columns)
    kata_ke_idx = daftar_kata if isinstance(daftar_kata, dict) else {kata: i for i, kata in enumerate(daftar_kata)}

    vektor = np.zeros((1, naive_bayes._vocab), dtype=np.float64)
    for kata in tokens:
        if kata in kata_ke_idx and kata_ke_idx[kata] < naive_bayes._vocab:
            vektor[0, kata_ke_idx[kata]] += 1.0
    return vektor

def prediksi_teks():
    teks = input_teks.get("1.0", tk.END).strip()
    if not teks:
        label_hasil.config(text="Silakan masukkan teks terlebih dahulu!")
        label_probabilitas.config(text="")
        return

    x_vektor = ubah_teks_ke_vektor(teks)
    kelas_prediksi = naive_bayes.predict(x_vektor)[0]
    probabilitas = naive_bayes.predict_probabilities(x_vektor)[0]
    keyakinan = np.max(probabilitas) * 100

    label_hasil.config(text=f"Hasil Prediksi: {kelas_prediksi} ({keyakinan:.2f}%)")

    urutan = np.argsort(probabilitas)[::-1]
    rincian = "\n".join([f"{naive_bayes._classes[i]} : {probabilitas[i] * 100:.2f}%" for i in urutan])
    label_probabilitas.config(text=rincian)

def reset_teks():
    input_teks.delete("1.0", tk.END)
    label_hasil.config(text="Hasil Prediksi: -")
    label_probabilitas.config(text="")

root = tk.Tk()
root.title("Klasifikasi Teks Naive Bayes Multinomial")
root.geometry("550x480")
root.configure(padx=15, pady=15)

tk.Label(root, text="KLASIFIKASI TEKS NAIVE BAYES", font=("Arial", 13, "bold")).pack(pady=(0, 5))

info_evaluasi = (
    f"Train Acc: {summary_train['Accuracy']*100:.2f}%  |  "
    f"Test Acc: {summary_test['Accuracy']*100:.2f}%  |  "
    f"Weighted F1: {summary_test['Weighted F1-Score']*100:.2f}%"
)
tk.Label(root, text=info_evaluasi, font=("Arial", 9)).pack(pady=(0, 10))

tk.Label(root, text="Masukkan Teks / Komentar:", font=("Arial", 10, "bold"), anchor="w").pack(fill="x")
input_teks = tk.Text(root, height=4, font=("Arial", 10), wrap="word")
input_teks.pack(fill="x", pady=5)

frame_tombol = tk.Frame(root)
frame_tombol.pack(pady=5)
tk.Button(frame_tombol, text="Prediksi", command=prediksi_teks, width=12, bg="#1e88e5", fg="white", font=("Arial", 10, "bold")).pack(side="left", padx=5)
tk.Button(frame_tombol, text="Reset", command=reset_teks, width=12, bg="#e53935", fg="white", font=("Arial", 10, "bold")).pack(side="left", padx=5)

label_hasil = tk.Label(root, text="Hasil Prediksi: -", font=("Arial", 11, "bold"), fg="#0d47a1", pady=8)
label_hasil.pack()

tk.Label(root, text="Probabilitas Tiap Kelas:", font=("Arial", 10, "bold")).pack()
label_probabilitas = tk.Label(root, text="", font=("Arial", 10), justify="left")
label_probabilitas.pack(pady=5)

root.mainloop()